In [ ]:
import pandas as pd
import osmnx as ox
import geopandas as gpd
from tqdm import tqdm
from functools import lru_cache

In [ ]:
df = pd.read_csv('data_migration/data_birth_death.csv')
df[(df.year == 2019) & (df.indicator_name == 'Численность постоянного населения на 1 января') & (df.area_type == 'городское и сельское население')][['object_name', 'year', 'indicator_value']]

C:\Users\Local\AppData\Local\Temp\ipykernel_13524\3690763627.py:1: DtypeWarning: Columns (7,8,10,11,12,13,14,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data_migration/data_birth_death.csv')


,id,object_name,object_oktmo,object_okato,year,indicator_name,indicator_unit,area_type,age_group,year_of_birth,sex,month,birth_order,mother_marital_status,death_cause,indicator_value,original_accuracy,methodology,notes
0,1,Алтайский край,1.000000e+09,1.000000e+09,1990,Численность постоянного населения на 1 января,человек,городское и сельское население,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2640419.0,0.0,population_2,NaN
1,2,Алтайский край,1.000000e+09,1.000000e+09,1991,Численность постоянного населения на 1 января,человек,городское и сельское население,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2653818.0,0.0,population_2,NaN
2,3,Алтайский край,1.000000e+09,1.000000e+09,1992,Численность постоянного населения на 1 января,человек,городское и сельское население,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2664886.0,0.0,population_2,NaN
3,4,Алтайский край,1.000000e+09,1.000000e+09,1993,Численность постоянного населения на 1 января,человек,городское и сельское население,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2680334.0,0.0,population_2,NaN
4,5,Алтайский край,1.000000e+09,1.000000e+09,1994,Численность постоянного населения на 1 января,человек,городское и сельское население,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2684297.0,0.0,population_2,NaN


,object_name,year,indicator_value
29,Алтайский край,2019,2332813.0
60,Амурская область,2019,793194.0
91,Архангельская область,2019,1144119.0
122,Архангельская область без Ненецкого автономног...,2019,1100290.0
153,Астраханская область,2019,1014065.0
...,...,...,...
2850,Чувашская Республика – Чувашия,2019,1223395.0
2881,Чукотский автономный округ,2019,49663.0
2912,Южный федеральный округ,2019,16454550.0
2943,Ямало-Ненецкий автономный округ,2019,541479.0


In [ ]:
gdf = gpd.read_file("optimized_regions.gpkg")
gdf.dropna(inplace=True)


In [ ]:
import osmnx as ox
import geopandas as gpd
from tqdm import tqdm

# Определение тегов OSM
tags_config = {
    'clinic': {'amenity': 'clinic'},
    'theatre': {'amenity': 'theatre'},
    'museum': {'tourism': 'museum'},
    'park': {'leisure': 'park'}
}

# Функция для подсчета объектов
def count_pois(polygon, tag):
    try:
        gdf = ox.features_from_polygon(
            polygon,
            tags=tag,
        )
        return gdf[gdf.geometry.notnull()].shape[0]
    except:
        return 0

# Добавляем колонки для каждого типа объектов
for category in tags_config:
    gdf[f'{category}'] = 0

# Обрабатываем все полигоны с визуализацией прогресса
for idx, row in tqdm(gdf.iterrows(), total=len(gdf)):
    geom = row.geometry
    for category, tag in tags_config.items():
        count = count_pois(geom, tag)
        print(f'Объектов "{category}": {count}')
        gdf.at[idx, f'{category}'] = count

In [ ]:
import osmnx as ox
import geopandas as gpd
from tqdm import tqdm
import concurrent.futures

# Определение тегов OSM
tags_config = {
    'clinic': {'amenity': 'clinic'},
    'theatre': {'amenity': 'theatre'},
    'museum': {'tourism': 'museum'},
    'park': {'leisure': 'park'}
}

# Функция для подсчета объектов
def count_pois(polygon):
    try:
        gdf = ox.features_from_polygon(polygon, tags=list(tags_config.values()))
        counts = {category: gdf[gdf.geometry.notnull() & (gdf['amenity'] == tag['amenity'])].shape[0] if 'amenity' in tag else
                  gdf[gdf.geometry.notnull() & (gdf['tourism'] == tag['tourism'])].shape[0] if 'tourism' in tag else
                  gdf[gdf.geometry.notnull() & (gdf['leisure'] == tag['leisure'])].shape[0] if 'leisure' in tag else 0
                  for category, tag in tags_config.items()}
        return counts
    except:
        return {category: 0 for category in tags_config}

# Добавляем колонки для каждого типа объектов
for category in tags_config:
    gdf[f'{category}'] = 0

# Обрабатываем все полигоны с визуализацией прогресса
def process_polygon(row):
    geom = row.geometry
    counts = count_pois(geom)
    for category, count in counts.items():
        print(f'Объектов "{category}": {count}')
        gdf.at[row.name, f'{category}'] = count

# Параллельная обработка
with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(process_polygon, row) for idx, row in gdf.iterrows()]
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures)):
        future.result()


In [ ]:
import osmnx as ox
import geopandas as gpd

tags_config = {
'tourism': True,
'leisure': True,
'healthcare': True
}

# Функция для подсчета объектов
def count_pois(polygon):
    try:
        gdf = ox.features_from_polygon(polygon, tags=tags_config)
        
        counts = {
            'clinic': gdf[(gdf['amenity'].notna())].shape[0],
            'theatre': gdf[(gdf['amenity'].notna())].shape[0],
            'museum': gdf[(gdf['tourism'].notna())].shape[0],
            'park': gdf[(gdf['leisure'].notna())].shape[0]
        }
        
        return counts
    except:
        return {category: 0 for category in tags_config}


# Пример использования
polygon = gdf.iloc[0].geometry
count = count_pois(polygon)
print(f'Объектов: {count}')

In [ ]:
import osmnx as ox
import geopandas as gpd

tags_config = {
    'tourism': True,
    'leisure': True,
    'healthcare': True
}

# Функция для подсчета объектов с кэшированием
cache = {}  # Словарь для кэширования результатов

def count_pois(polygon):
    global cache  # Используем глобальный словарь для кэширования
    
    # Проверяем, есть ли результат в кэше
    if polygon.wkt in cache:
        return cache[polygon.wkt]
    
    try:
        gdf = ox.features_from_polygon(polygon, tags=tags_config)
        
        counts = {
            'tourism': gdf[(gdf['tourism'].notna())].shape[0],
            'leisure': gdf[(gdf['leisure'].notna())].shape[0],
            'healthcare': gdf[(gdf['healthcare'].notna())].shape[0],
        }
        
        # Сохраняем результат в кэше
        cache[polygon.wkt] = counts
        
        return counts
    except:
        # Если ошибка, сохраняем пустой словарь в кэше
        cache[polygon.wkt] = {category: 0 for category in ['tourism', 'leisure', 'healthcare']}
        return cache[polygon.wkt]

# Добавляем колонки для каждого типа объектов
gdf['tourism'] = 0
gdf['leisure'] = 0
gdf['healthcare'] = 0

# Обрабатываем все полигоны с визуализацией прогресса
for idx, row in gdf.iterrows():
    geom = row.geometry
    counts = count_pois(geom)
    
    # Обновляем значения в gdf
    gdf.at[idx, 'tourism'] = counts['tourism']
    gdf.at[idx, 'leisure'] = counts['leisure']
    gdf.at[idx, 'healthcare'] = counts['healthcare']
    
    print(f'Объектов для полигона {idx}: {counts}')

In [ ]:
from shapely import Point

In [ ]:
gdf = gdf[~(gdf.geometry.geom_type == 'Point')]

In [ ]:
output_file = "russia.gpkg"
gdf.to_file(output_file, driver='GPKG')

## NY

In [ ]:
import skmob
from skmob.utils import utils, constants
from skmob.models.gravity import Gravity

import numpy as np
import pandas as pd
import geopandas as gpd

import osmnx as ox
from shapely.geometry import Polygon

url_tess = skmob.utils.constants.NY_COUNTIES_2011
tessellation = gpd.read_file(url_tess).rename(columns={'tile_id': 'tile_ID'})

fdf = skmob.FlowDataFrame.from_file(skmob.utils.constants.NY_FLOWS_2011,
                                        tessellation=tessellation,
                                        tile_id='tile_ID',
                                        sep=",")
                      
tot_outflows = fdf[fdf['origin'] != fdf['destination']].groupby(by='destination', axis=0)[['flow']].sum().fillna(0) # Группировка по городам и подсчет миграции в каждый город
tessellation = tessellation.merge(tot_outflows, left_on='tile_ID', right_on='destination').rename(columns={'flow': constants.TOT_OUTFLOW}) # Merge с городами, населением и геометрией

In [ ]:
import osmnx as ox
import geopandas as gpd

tags_config = {
    'tourism': True,
    'leisure': True,
    'healthcare': True
}

# Функция для подсчета объектов с кэшированием
cache = {}  # Словарь для кэширования результатов

def count_pois(polygon):
    global cache  # Используем глобальный словарь для кэширования
    
    # Проверяем, есть ли результат в кэше
    if polygon.wkt in cache:
        return cache[polygon.wkt]
    
    try:
        tessellation = ox.features_from_polygon(polygon, tags=tags_config)
        
        counts = {
            'tourism': tessellation[(tessellation['tourism'].notna())].shape[0],
            'leisure': tessellation[(tessellation['leisure'].notna())].shape[0],
            'healthcare': tessellation[(tessellation['healthcare'].notna())].shape[0],
        }
        
        # Сохраняем результат в кэше
        cache[polygon.wkt] = counts
        
        return counts
    except:
        # Если ошибка, сохраняем пустой словарь в кэше
        cache[polygon.wkt] = {category: 0 for category in ['tourism', 'leisure', 'healthcare']}
        return cache[polygon.wkt]

# Добавляем колонки для каждого типа объектов
tessellation['tourism'] = 0
tessellation['leisure'] = 0
tessellation['healthcare'] = 0

# Обрабатываем все полигоны с визуализацией прогресса
for idx, row in tessellation.iterrows():
    geom = row.geometry
    counts = count_pois(geom)
    
    # Обновляем значения в tessellation
    tessellation.at[idx, 'tourism'] = counts['tourism']
    tessellation.at[idx, 'leisure'] = counts['leisure']
    tessellation.at[idx, 'healthcare'] = counts['healthcare']
    
    print(f'Объектов для полигона {idx}: {counts}')

output_file = "NY.gpkg"
tessellation.to_file(output_file, driver='GPKG')